# Lesson 4: Transfer Learning

**Duration:** 75 minutes

## Learning Objectives
- Understand why transfer learning is powerful for computer vision
- Load and use pretrained models from torchvision
- Fine-tune pretrained models on custom datasets
- Understand freezing/unfreezing layers and learning rate schedules
- Compare transfer learning vs. training from scratch

## 4.1 What is Transfer Learning?

**Transfer Learning:** Use a model pretrained on a large dataset (e.g., ImageNet) and adapt it to your task.

**Advantages:**
- Faster training (fewer epochs needed)
- Better performance with limited data
- Leverages features learned from millions of images
- Reduces computational cost

**Strategies:**
1. **Feature Extraction:** Freeze pretrained weights, train only the head
2. **Fine-tuning:** Train all layers with a low learning rate

## 4.2 Loading Pretrained Models

Torchvision provides many pretrained models:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from tqdm import tqdm
import matplotlib.pyplot as plt

# Load a pretrained ResNet-18
print('Available pretrained models:')
print('resnet18, resnet50, vgg16, alexnet, squeezenet, densenet, inception, mobilenet...')

# Load model
model_pretrained = models.resnet18(pretrained=True)
print(f'\nLoaded ResNet-18 from ImageNet')
print(f'Total parameters: {sum(p.numel() for p in model_pretrained.parameters()) / 1e6:.2f}M')

# Check final layer
print(f'\nFinal layer: {model_pretrained.fc}')
print(f'Output classes from ImageNet: 1000')

## 4.3 Adapting for Our Task

Modify the head for CIFAR-10 (10 classes):

In [ ]:
# Modify the final layer for CIFAR-10
num_classes = 10
in_features = model_pretrained.fc.in_features
model_pretrained.fc = nn.Linear(in_features, num_classes)

print(f'Modified final layer: {model_pretrained.fc}')
print(f'Now outputs {num_classes} classes')

# Transfer to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model_pretrained.to(device)
print(f'\nModel transferred to device: {device}')

## 4.4 Strategy 1: Feature Extraction (Frozen Backbone)

Freeze all layers except the head:

In [ ]:
# Strategy 1: Freeze backbone, train only head
for name, param in model.named_parameters():
    if 'fc' not in name:  # Freeze all layers except the final layer
        param.requires_grad = False

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f'Total parameters: {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters: {(total_params - trainable_params) / 1e6:.2f}M')
print(f'\nTraining only {(trainable_params / total_params * 100):.1f}% of parameters!')

# Create optimizer for trainable parameters only
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

print(f'\nOptimizer configured to update only trainable parameters')

## 4.5 Prepare Data and Train

Load CIFAR-10 and train the model:

In [ ]:
# Load data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

# Train for a few epochs
num_epochs = 2
train_losses = []
test_accuracies = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Test
    model.eval()
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
    
    test_acc = test_correct / test_total
    test_accuracies.append(test_acc)
    
    print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}')

print(f'\nFinal test accuracy with feature extraction: {test_accuracies[-1]:.4f}')

## 4.6 Strategy 2: Fine-tuning All Layers

Unfreeze all layers and use a smaller learning rate:

In [ ]:
# Load fresh model for fine-tuning
model_finetune = models.resnet18(pretrained=True)
model_finetune.fc = nn.Linear(model_finetune.fc.in_features, 10)
model_finetune = model_finetune.to(device)

# Unfreeze all layers
for param in model_finetune.parameters():
    param.requires_grad = True

# Use much smaller learning rate for fine-tuning
optimizer_ft = optim.Adam(model_finetune.parameters(), lr=0.00001)

print('Fine-tuning strategy:')
print(f'  - Learning rate: 0.00001 (much smaller than feature extraction)')
print(f'  - All layers trainable')
print(f'  - Fewer epochs typically needed')

# Quick fine-tuning on 1 epoch for comparison
model_finetune.train()
epoch_loss_ft = 0.0
correct_ft = 0
total_ft = 0

for images, labels in tqdm(train_loader, desc='Fine-tuning epoch 1'):
    images, labels = images.to(device), labels.to(device)
    
    outputs = model_finetune(images)
    loss = criterion(outputs, labels)
    
    optimizer_ft.zero_grad()
    loss.backward()
    optimizer_ft.step()
    
    epoch_loss_ft += loss.item()
    _, predicted = torch.max(outputs, 1)
    total_ft += labels.size(0)
    correct_ft += (predicted == labels).sum().item()

print(f'\nFine-tuning loss: {epoch_loss_ft / len(train_loader):.4f}')
print(f'Fine-tuning accuracy: {correct_ft / total_ft:.4f}')